In [ ]:
import requests
import time
from tqdm import tqdm
from urllib.parse import unquote
import pandas as pd


df = pd.read_csv("politicians_1900_1950.csv")


def fetch_wikipedia_text(url: str, session=None) -> str:
    """
    Fetch plain text from a Wikipedia article using the MediaWiki extracts API.
    Returns the article extract as plain text.
    """
    if not isinstance(url, str) or "/wiki/" not in url:
        return ""

    if session is None:
        session = requests.Session()
        session.headers.update({"User-Agent": "PoliticiansNetwork/1.0"})

    title = unquote(url.rstrip("/").split("/wiki/")[-1])

    api_url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "titles": title,
        "prop": "extracts",
        "explaintext": True,
        "exsectionformat": "plain",
        "format": "json",
    }

    r = session.get(api_url, params=params, timeout=20)
    r.raise_for_status()

    pages = r.json().get("query", {}).get("pages", {})
    if not pages:
        return ""

    page = next(iter(pages.values()))
    return page.get("extract", "") or ""


def build_text_corpus(df, max_articles=None, delay=0.3):
    """
    Download article text for rows with a valid wikipedia_url.
    Returns a dict {wikidata_qid: text}.
    """
    corpus = {}
    session = requests.Session()
    session.headers.update({"User-Agent": "PoliticiansNetwork/1.0"})

    subset = df[df["wikipedia_url"].notna() & (df["wikipedia_url"] != "")].copy()

    if max_articles is not None:
        subset = subset.head(max_articles)

    for _, row in tqdm(subset.iterrows(), total=len(subset), desc="Fetching article text"):
        try:
            text = fetch_wikipedia_text(row["wikipedia_url"], session=session)
            corpus[row["wikidata"]] = text
        except Exception as e:
            print(f"Error fetching {row['wikipedia_url']}: {e}")
            corpus[row["wikidata"]] = ""

        time.sleep(delay)

    return corpus


# =========================
# RUN EVERYTHING
# =========================

# Hvis df ikke allerede findes, så load jeres CSV:
# df = pd.read_csv("politicians_1900_1950.csv")

# Hent tekst for alle politikere
corpus = build_text_corpus(df, max_articles=20, delay=0.3)

# Tilføj ny kolonne til dataframe
df["article_text"] = df["wikidata"].map(corpus)

# Erstat NaN med tom streng
df["article_text"] = df["article_text"].fillna("")

# Gem ny CSV med tekst-kolonnen
output_csv = "politicians_with_text.csv"
df.to_csv(output_csv, index=False)

# Kort status
num_nonempty = (df["article_text"].str.len() > 0).sum()
print(f"Downloaded non-empty text for {num_nonempty} politicians")
print(f"Saved updated dataframe to: {output_csv}")

# Vis eksempel
print(df[["name", "wikipedia_url", "article_text"]].head(3).to_string())

# Hvis grafen findes, læg teksten på noderne også
if "G_lcc" in globals():
    for qid, text in corpus.items():
        if qid in G_lcc:
            G_lcc.nodes[qid]["article_text"] = text

    print("\nArticle text stored on network nodes.")

print("Ready for TF-IDF, topic modelling, NER, embeddings, etc.")